<a href="https://colab.research.google.com/github/AlifHammam/data-science-2026/blob/main/Pertemuan3_AlifHamammamMultazam_240401010043.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pertemuan 3 - Data Cleaning: Missing Values, Outlier & Ekstraksi Data

**Nama Lengkap:** Alif Hamammam Multazam  
**NIM:** 240401010043  
**Kelas:** IF403

## 1. Import Library

In [47]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats.mstats import winsorize
import requests
from pandas import json_normalize

print('Library berhasil diimport!')

Library berhasil diimport!


## 2. Load Dataset & Eksplorasi Awal

In [48]:
# Load dataset housing_dirty.csv dari Google Drive
file_id = "1LfQWProB0VjWN5q8bKuRIgn-stULfIRo"
url = f"https://drive.google.com/uc?id={file_id}"

df = pd.read_csv(url)

print("Shape awal:", df.shape)
df.head(10)

Shape awal: (130, 7)


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang
5,6,153.3,814.0,Jakarta,1.0,2006,Sedang
6,7,114.3,NaN,jakarta,3.0,2011,baik
7,8,NaN,333.0,Yogyakarta,NaN,1989,baik sekali
8,9,81.2,307.0,Yogyakarta,1.0,1996,SEDANG
9,10,69.1,237.0,Bandung,5.0,1980,baik


In [49]:
# Informasi umum dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB


In [50]:
df.describe()

,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000


In [51]:
# Cek missing values
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Jumlah Missing': missing, 'Proporsi (%)': missing_pct})
print(missing_df[missing_df['Jumlah Missing'] > 0])

            Jumlah Missing  Proporsi (%)
luas_m2                 18         13.85
harga_juta              17         13.08
kamar                   10          7.69


## 3. Pipeline Data Cleaning

In [52]:
# STEP 1 — Hapus Duplikat
n_dup = df.duplicated().sum()
print(f'Baris duplikat ditemukan: {n_dup}')

df.drop_duplicates(inplace=True)
print('Setelah hapus duplikat:', df.shape)

Baris duplikat ditemukan: 0
Setelah hapus duplikat: (130, 7)


In [53]:
# STEP 2 — Normalisasi String
# Sebelum normalisasi
print('Nilai unik kota (sebelum):', df['kota'].unique())
print('Nilai unik kondisi (sebelum):', df['kondisi'].unique())

df['kota']    = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

print('\nNilai unik kota (sesudah):', df['kota'].unique())
print('Nilai unik kondisi (sesudah):', df['kondisi'].unique())

Nilai unik kota (sebelum): ['jogja' 'Medan' 'Depok' 'YGY' 'Jakarta' 'jakarta' 'Yogyakarta' 'Bandung'
 'Surabaya' 'dpk' 'sby' 'Makassar' 'mdn' 'medan' 'Semarang' 'semarang'
 'yogyakarta' 'Jogja' 'JAKARTA' 'Smg' 'DEPOK' 'Bdg' 'makassar' 'surabaya'
 'MAKASSAR' 'depok' 'bandung' 'Bandung ' 'SURABAYA' 'Mksr' ' Jakarta']
Nilai unik kondisi (sebelum): ['baik' 'Bagus' 'Sedang' 'baik sekali' 'SEDANG' 'sedang' 'BAIK' 'rusak'
 'cukup' 'Baik' 'Cukup' 'perlu renovasi' 'bagus' 'jelek' 'RUSAK']

Nilai unik kota (sesudah): ['Jogja' 'Medan' 'Depok' 'Ygy' 'Jakarta' 'Yogyakarta' 'Bandung' 'Surabaya'
 'Dpk' 'Sby' 'Makassar' 'Mdn' 'Semarang' 'Smg' 'Bdg' 'Mksr']
Nilai unik kondisi (sesudah): ['baik' 'bagus' 'sedang' 'baik sekali' 'rusak' 'cukup' 'perlu renovasi'
 'jelek']


In [54]:
# STEP 3 — Imputasi Missing Values
# Numerik pakai median (lebih robust terhadap outlier)
df['luas_m2']    = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())

# Kategorik pakai modus
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

print('Missing setelah imputasi:')
print(df.isnull().sum())

Missing setelah imputasi:
id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64


In [55]:
# STEP 4 — Tangani Outlier dengan IQR Fence (Capping)
def deteksi_outlier_iqr(df, kolom):
    Q1 = df[kolom].quantile(0.25)
    Q3 = df[kolom].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[kolom] < lower) | (df[kolom] > upper)]
    return lower, upper, outliers

for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    lower, upper, out_df = deteksi_outlier_iqr(df, col)
    print(f'[{col}] Outlier: {len(out_df)} baris | Batas: [{lower:.2f}, {upper:.2f}]')
    df[col] = df[col].clip(lower=lower, upper=upper)

[harga_juta] Outlier: 3 baris | Batas: [-422.75, 1719.25]
[luas_m2] Outlier: 1 baris | Batas: [-145.22, 512.97]
[tahun_bangun] Outlier: 3 baris | Batas: [1960.50, 2042.50]


In [56]:
# STEP 5 — Validasi Akhir
assert df.isnull().sum().sum() == 0, 'Masih ada missing values!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'

print('Validasi berhasil!')
print('Shape akhir:', df.shape)
df.head()

Validasi berhasil!
Shape akhir: (130, 7)


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,Jogja,2.0,2000.0,baik
1,2,254.0,761.0,Medan,1.0,1995.0,bagus
2,3,249.7,895.0,Depok,1.0,1983.0,baik
3,4,49.7,178.0,Ygy,5.0,2013.0,baik
4,5,133.4,424.0,Medan,5.0,2004.0,sedang


In [57]:
# Ekspor dataset yang sudah bersih
df.to_csv('housing_clean.csv', index=False)
print('Dataset bersih berhasil disimpan ke housing_clean.csv')

Dataset bersih berhasil disimpan ke housing_clean.csv


## 4. Ekstraksi Data dari REST API (JSONPlaceholder)

In [58]:
# Akses API JSONPlaceholder — ambil data users
URL = "https://jsonplaceholder.typicode.com/users"

response = requests.get(URL, timeout=10)

if response.status_code == 200:
    data = response.json()
    df_users = json_normalize(data, sep='_')
    print('Data berhasil diambil!')
    print('Shape:', df_users.shape)
    print(df_users[['id', 'name', 'email', 'address_city']].head())
else:
    print(f'Error: {response.status_code}')

Data berhasil diambil!
Shape: (10, 15)
   id              name                      email   address_city
0   1     Leanne Graham          Sincere@april.biz    Gwenborough
1   2      Ervin Howell          Shanna@melissa.tv    Wisokyburgh
2   3  Clementine Bauch         Nathan@yesenia.net  McKenziehaven
3   4  Patricia Lebsack  Julianne.OConner@kory.org    South Elvis
4   5  Chelsey Dietrich   Lucio_Hettinger@annie.ca     Roscoeview


In [59]:
# Akses API dengan parameter — ambil posts dari user tertentu
params = {'userId': 1}
response_posts = requests.get(
    "https://jsonplaceholder.typicode.com/posts",
    params=params,
    timeout=10
)

if response_posts.status_code == 200:
    df_posts = pd.DataFrame(response_posts.json())
    print('Posts dari userId=1:')
    print(df_posts[['id', 'title']].head())
else:
    print(f'Error: {response_posts.status_code}')

Posts dari userId=1:
   id                                              title
0   1  sunt aut facere repellat provident occaecati e...
1   2                                       qui est esse
2   3  ea molestias quasi exercitationem repellat qui...
3   4                               eum et est occaecati
4   5                                 nesciunt quas odio


## Kesimpulan

Pada praktikum pertemuan ketiga ini saya mempelajari proses data cleaning yang mencakup penanganan missing values, penghapusan data duplikat, normalisasi string, serta deteksi dan penanganan outlier menggunakan metode IQR Fence. Selain itu, saya juga mempelajari cara mengekstrak data dari REST API publik menggunakan library requests dan mengonversinya ke dalam DataFrame Pandas.

Melalui dataset housing_dirty.csv, saya memahami secara langsung bagaimana data nyata bisa mengandung berbagai masalah kualitas seperti nilai yang hilang, duplikasi baris, inkonsistensi format string, dan nilai ekstrem yang tidak wajar. Pipeline cleaning yang dijalankan secara berurutan sangat membantu agar proses pembersihan data lebih terstruktur dan dapat direproduksi.

Keterbatasan pada praktikum ini adalah penanganan outlier masih menggunakan pendekatan capping yang sederhana, dan belum mempertimbangkan konteks bisnis dari setiap nilai ekstrem yang ditemukan. Namun, praktikum ini memberikan pemahaman yang baik tentang pentingnya kualitas data sebelum masuk ke tahap analisis atau pemodelan lebih lanjut.